# Phase 2 - Notebook 03: Gaussian Map Initialization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/03_map_initialization.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why initialization matters for 3DGS-SLAM
2. Learn depth-guided Gaussian placement
3. Master initial scale and opacity estimation
4. Understand color initialization from RGB
5. Compare RGB-D vs monocular initialization

**Estimated Time**: 60 minutes

**Prerequisites**: Notebook 02 (SplaTAM Architecture)

---

## 1. Why Initialization Matters

In offline 3DGS training, we initialize from COLMAP point clouds. In SLAM, we must:

1. **Initialize from scratch** - No pre-computed structure
2. **Initialize online** - Must work in real-time
3. **Initialize incrementally** - Add Gaussians as new areas are observed

### Challenges

| Challenge | Impact | Solution |
|-----------|--------|----------|
| No COLMAP | No initial points | Use depth |
| Unknown scale | Wrong Gaussian sizes | Depth-based estimation |
| Sparse coverage | Holes in rendering | Dense depth sampling |
| Online operation | Limited time | Efficient initialization |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

print("Gaussian Map Initialization Tutorial")
print("=" * 40)

## 2. Camera Model Review

Before initializing Gaussians, let's review the camera model.

### Pinhole Camera Model

**Projection** (3D → 2D):
$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \frac{1}{z} K \begin{bmatrix} x \\ y \\ z \end{bmatrix}
$$

**Unprojection** (2D + depth → 3D):
$$
\begin{bmatrix} x \\ y \\ z \end{bmatrix} = z \cdot K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
$$

Where:
- $(u, v)$: pixel coordinates
- $(x, y, z)$: 3D point in camera frame
- $K$: intrinsic matrix

In [ ]:
@dataclass
class CameraIntrinsics:
    """Camera intrinsic parameters."""
    fx: float  # Focal length x
    fy: float  # Focal length y
    cx: float  # Principal point x
    cy: float  # Principal point y
    width: int
    height: int
    
    def to_matrix(self) -> torch.Tensor:
        """Get 3x3 intrinsic matrix K."""
        return torch.tensor([
            [self.fx, 0, self.cx],
            [0, self.fy, self.cy],
            [0, 0, 1]
        ], dtype=torch.float32)
    
    def to_inverse_matrix(self) -> torch.Tensor:
        """Get inverse intrinsic matrix K^{-1}."""
        return torch.tensor([
            [1/self.fx, 0, -self.cx/self.fx],
            [0, 1/self.fy, -self.cy/self.fy],
            [0, 0, 1]
        ], dtype=torch.float32)


def project_points(
    points_3d: torch.Tensor,  # [N, 3]
    intrinsics: CameraIntrinsics,
    pose: torch.Tensor = None,  # [4, 4] world-to-camera
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Project 3D points to 2D pixel coordinates.
    
    Returns:
        pixels: [N, 2] pixel coordinates (u, v)
        depths: [N] depth values
    """
    # Transform to camera frame if pose given
    if pose is not None:
        R = pose[:3, :3]
        t = pose[:3, 3]
        points_cam = (R @ points_3d.T).T + t
    else:
        points_cam = points_3d
    
    # Project
    x, y, z = points_cam[:, 0], points_cam[:, 1], points_cam[:, 2]
    
    u = intrinsics.fx * x / z + intrinsics.cx
    v = intrinsics.fy * y / z + intrinsics.cy
    
    return torch.stack([u, v], dim=1), z


def unproject_depth(
    depth: torch.Tensor,  # [H, W]
    intrinsics: CameraIntrinsics,
    pose: torch.Tensor = None,  # [4, 4] world-to-camera
) -> torch.Tensor:
    """
    Unproject depth map to 3D points.
    
    Args:
        depth: [H, W] depth map
        intrinsics: Camera intrinsics
        pose: Optional camera pose (world-to-camera)
    
    Returns:
        points: [H*W, 3] 3D points (in world frame if pose given)
    """
    H, W = depth.shape
    device = depth.device
    
    # Create pixel grid
    v, u = torch.meshgrid(
        torch.arange(H, device=device, dtype=torch.float32),
        torch.arange(W, device=device, dtype=torch.float32),
        indexing='ij'
    )
    
    # Unproject to camera coordinates
    z = depth
    x = (u - intrinsics.cx) * z / intrinsics.fx
    y = (v - intrinsics.cy) * z / intrinsics.fy
    
    points_cam = torch.stack([x, y, z], dim=-1)  # [H, W, 3]
    points_cam = points_cam.reshape(-1, 3)  # [H*W, 3]
    
    # Transform to world frame if pose given
    if pose is not None:
        pose_inv = torch.linalg.inv(pose)
        R = pose_inv[:3, :3]
        t = pose_inv[:3, 3]
        points_world = (R @ points_cam.T).T + t
        return points_world
    
    return points_cam


# Create test camera
camera = CameraIntrinsics(
    fx=500, fy=500,
    cx=320, cy=240,
    width=640, height=480
)

print("Camera Intrinsics:")
print(f"  Focal length: ({camera.fx}, {camera.fy})")
print(f"  Principal point: ({camera.cx}, {camera.cy})")
print(f"  Image size: {camera.width} x {camera.height}")
print(f"\nIntrinsic matrix K:\n{camera.to_matrix()}")

## 3. Depth-Guided Gaussian Placement

The core idea: **Every pixel with valid depth becomes a Gaussian**.

### Process

1. For each pixel $(u, v)$ with depth $d > 0$:
2. Unproject to 3D: $(x, y, z) = d \cdot K^{-1} (u, v, 1)^T$
3. Transform to world frame: $p_{world} = T_{c2w} \cdot p_{cam}$
4. Create Gaussian at $p_{world}$

In [ ]:
def create_synthetic_rgbd(
    intrinsics: CameraIntrinsics,
    scene_type: str = "box",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Create a synthetic RGB-D frame.
    
    Returns:
        rgb: [3, H, W] RGB image
        depth: [H, W] depth map
    """
    H, W = intrinsics.height, intrinsics.width
    
    # Initialize
    rgb = torch.zeros(3, H, W)
    depth = torch.zeros(H, W)
    
    if scene_type == "box":
        # Create a box scene
        # Back wall at 3m
        depth[:] = 3.0
        rgb[:] = 0.3  # Gray background
        
        # Left wall
        for v in range(H):
            for u in range(W // 4):
                # Wall slanting from left edge
                d = 1.0 + 2.0 * u / (W // 4)
                depth[v, u] = d
                rgb[0, v, u] = 0.6  # Red tint
        
        # Right wall
        for v in range(H):
            for u in range(3 * W // 4, W):
                d = 1.0 + 2.0 * (W - 1 - u) / (W // 4)
                depth[v, u] = d
                rgb[2, v, u] = 0.6  # Blue tint
        
        # Floor
        for v in range(3 * H // 4, H):
            for u in range(W // 4, 3 * W // 4):
                d = 1.5 + 1.5 * (v - 3 * H // 4) / (H // 4)
                depth[v, u] = d
                rgb[1, v, u] = 0.5  # Green tint
        
        # Object in center
        obj_y, obj_x = H // 2, W // 2
        obj_size = 50
        for v in range(obj_y - obj_size, obj_y + obj_size):
            for u in range(obj_x - obj_size, obj_x + obj_size):
                if 0 <= v < H and 0 <= u < W:
                    # Sphere-like depth
                    dy = (v - obj_y) / obj_size
                    dx = (u - obj_x) / obj_size
                    r2 = dx*dx + dy*dy
                    if r2 < 1:
                        dz = np.sqrt(1 - r2) * 0.3
                        depth[v, u] = 1.5 - dz
                        rgb[:, v, u] = torch.tensor([0.8, 0.2, 0.2])
    
    elif scene_type == "plane":
        # Simple plane at 2m
        depth[:] = 2.0
        # Checkerboard pattern
        for v in range(H):
            for u in range(W):
                if (v // 40 + u // 40) % 2 == 0:
                    rgb[:, v, u] = 0.8
                else:
                    rgb[:, v, u] = 0.2
    
    return rgb, depth


# Create synthetic scene
rgb, depth = create_synthetic_rgbd(camera, scene_type="box")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(rgb.permute(1, 2, 0).numpy())
axes[0].set_title('RGB Image')
axes[0].axis('off')

im = axes[1].imshow(depth.numpy(), cmap='viridis')
axes[1].set_title('Depth Map')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], label='Depth (m)')

plt.tight_layout()
plt.show()

print(f"RGB shape: {rgb.shape}")
print(f"Depth range: [{depth.min():.2f}, {depth.max():.2f}] m")

In [ ]:
# Unproject depth to 3D points

def initialize_gaussians_from_rgbd(
    rgb: torch.Tensor,  # [3, H, W]
    depth: torch.Tensor,  # [H, W]
    intrinsics: CameraIntrinsics,
    pose: torch.Tensor = None,  # [4, 4] world-to-camera
    subsample: int = 4,
    min_depth: float = 0.1,
    max_depth: float = 10.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Initialize Gaussian positions and colors from RGB-D frame.
    
    Args:
        rgb: [3, H, W] RGB image in [0, 1]
        depth: [H, W] depth map
        intrinsics: Camera intrinsics
        pose: Camera pose (world-to-camera), None = identity
        subsample: Subsampling factor for pixels
        min_depth, max_depth: Valid depth range
    
    Returns:
        positions: [N, 3] Gaussian positions
        colors: [N, 3] Gaussian colors
        pixel_coords: [N, 2] Original pixel coordinates
    """
    H, W = depth.shape
    device = depth.device
    
    # Create pixel grid (subsampled)
    v_coords = torch.arange(0, H, subsample, device=device, dtype=torch.float32)
    u_coords = torch.arange(0, W, subsample, device=device, dtype=torch.float32)
    v_grid, u_grid = torch.meshgrid(v_coords, u_coords, indexing='ij')
    
    # Sample depth and RGB
    v_idx = v_grid.long()
    u_idx = u_grid.long()
    
    depth_sampled = depth[v_idx, u_idx]
    rgb_sampled = rgb[:, v_idx, u_idx]  # [3, h, w]
    
    # Flatten
    v_flat = v_grid.reshape(-1)
    u_flat = u_grid.reshape(-1)
    z = depth_sampled.reshape(-1)
    colors = rgb_sampled.reshape(3, -1).T  # [N, 3]
    
    # Filter valid depth
    valid = (z > min_depth) & (z < max_depth)
    v_flat = v_flat[valid]
    u_flat = u_flat[valid]
    z = z[valid]
    colors = colors[valid]
    
    # Unproject to camera coordinates
    x = (u_flat - intrinsics.cx) * z / intrinsics.fx
    y = (v_flat - intrinsics.cy) * z / intrinsics.fy
    
    points_cam = torch.stack([x, y, z], dim=1)  # [N, 3]
    
    # Transform to world frame
    if pose is not None:
        pose_inv = torch.linalg.inv(pose)
        R = pose_inv[:3, :3]
        t = pose_inv[:3, 3]
        positions = (R @ points_cam.T).T + t
    else:
        positions = points_cam
    
    pixel_coords = torch.stack([u_flat, v_flat], dim=1)
    
    return positions, colors, pixel_coords


# Initialize Gaussians
positions, colors, pixels = initialize_gaussians_from_rgbd(
    rgb, depth, camera, subsample=4
)

print(f"Initialized {len(positions)} Gaussians")
print(f"Position range: X=[{positions[:, 0].min():.2f}, {positions[:, 0].max():.2f}]")
print(f"               Y=[{positions[:, 1].min():.2f}, {positions[:, 1].max():.2f}]")
print(f"               Z=[{positions[:, 2].min():.2f}, {positions[:, 2].max():.2f}]")

In [ ]:
# Visualize 3D point cloud

fig = plt.figure(figsize=(14, 6))

# 3D view
ax1 = fig.add_subplot(121, projection='3d')

# Subsample for visualization
vis_idx = np.random.choice(len(positions), min(5000, len(positions)), replace=False)
pos_vis = positions[vis_idx].numpy()
col_vis = colors[vis_idx].numpy()

ax1.scatter(pos_vis[:, 0], pos_vis[:, 2], pos_vis[:, 1], 
           c=col_vis, s=1, alpha=0.5)

# Camera position (at origin looking along +Z)
ax1.scatter([0], [0], [0], c='red', s=100, marker='^', label='Camera')
ax1.quiver(0, 0, 0, 0, 0.5, 0, color='red', arrow_length_ratio=0.3)

ax1.set_xlabel('X')
ax1.set_ylabel('Z')
ax1.set_zlabel('Y')
ax1.set_title('Initialized Gaussians (3D View)')
ax1.legend()

# Top-down view
ax2 = fig.add_subplot(122)
ax2.scatter(pos_vis[:, 0], pos_vis[:, 2], c=col_vis, s=1, alpha=0.5)
ax2.scatter([0], [0], c='red', s=100, marker='^', label='Camera')
ax2.arrow(0, 0, 0, 0.5, head_width=0.1, head_length=0.05, fc='red', ec='red')
ax2.set_xlabel('X')
ax2.set_ylabel('Z')
ax2.set_title('Top-Down View')
ax2.set_aspect('equal')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Scale Estimation

Gaussian scale determines how large each Gaussian "splat" is. In SLAM, we estimate scales from:

1. **Depth gradient** - Areas with rapid depth change need smaller scales
2. **Pixel spacing** - At farther depths, pixels cover more area
3. **Local point density** - Sparse areas need larger scales

### Depth-Based Scale Estimation

For a pixel at depth $d$, the 3D footprint is approximately:
$$
\text{scale} = \frac{d}{f} \cdot \text{pixel\_size}
$$

Where $f$ is focal length and pixel_size accounts for subsampling.

In [ ]:
def estimate_gaussian_scales(
    depth: torch.Tensor,  # [H, W]
    intrinsics: CameraIntrinsics,
    subsample: int = 4,
    scale_factor: float = 1.0,
) -> torch.Tensor:
    """
    Estimate Gaussian scales from depth.
    
    Uses the depth to estimate how large each Gaussian should be
    to cover its pixel footprint in 3D.
    
    Args:
        depth: [H, W] depth map
        intrinsics: Camera intrinsics
        subsample: Pixel subsampling factor
        scale_factor: Multiplier for final scale
    
    Returns:
        scales: [N, 3] isotropic scales for each Gaussian
    """
    H, W = depth.shape
    
    # Sample depth at subsampled positions
    v_coords = torch.arange(0, H, subsample, dtype=torch.float32)
    u_coords = torch.arange(0, W, subsample, dtype=torch.float32)
    v_grid, u_grid = torch.meshgrid(v_coords, u_coords, indexing='ij')
    
    depth_sampled = depth[v_grid.long(), u_grid.long()]
    z = depth_sampled.reshape(-1)
    
    # Compute 3D footprint of a pixel
    # At depth z, one pixel covers z/f meters
    # With subsampling, it covers subsample * z/f meters
    pixel_footprint = subsample * z / intrinsics.fx
    
    # Scale should be roughly half the pixel footprint
    # (Gaussian sigma to cover the pixel)
    scale = pixel_footprint * scale_factor * 0.5
    
    # Isotropic scales (same in all directions)
    scales = scale.unsqueeze(-1).expand(-1, 3)
    
    return scales


def estimate_scales_from_depth_gradient(
    depth: torch.Tensor,  # [H, W]
    intrinsics: CameraIntrinsics,
    subsample: int = 4,
    min_scale: float = 0.001,
    max_scale: float = 0.1,
) -> torch.Tensor:
    """
    Estimate scales using depth gradient for adaptive sizing.
    
    Areas with high depth gradient (edges) get smaller scales.
    """
    H, W = depth.shape
    
    # Compute depth gradients
    grad_x = torch.zeros_like(depth)
    grad_y = torch.zeros_like(depth)
    
    grad_x[:, 1:] = depth[:, 1:] - depth[:, :-1]
    grad_y[1:, :] = depth[1:, :] - depth[:-1, :]
    
    grad_magnitude = torch.sqrt(grad_x**2 + grad_y**2)
    
    # Sample at subsampled positions
    v_coords = torch.arange(0, H, subsample, dtype=torch.long)
    u_coords = torch.arange(0, W, subsample, dtype=torch.long)
    v_grid, u_grid = torch.meshgrid(v_coords, u_coords, indexing='ij')
    
    depth_sampled = depth[v_grid, u_grid].reshape(-1)
    grad_sampled = grad_magnitude[v_grid, u_grid].reshape(-1)
    
    # Base scale from depth
    base_scale = subsample * depth_sampled / intrinsics.fx * 0.5
    
    # Reduce scale where gradient is high
    grad_factor = 1.0 / (1.0 + grad_sampled * 10)
    
    scale = base_scale * grad_factor
    scale = torch.clamp(scale, min_scale, max_scale)
    
    return scale.unsqueeze(-1).expand(-1, 3)


# Estimate scales
scales_basic = estimate_gaussian_scales(depth, camera, subsample=4)
scales_adaptive = estimate_scales_from_depth_gradient(depth, camera, subsample=4)

# Filter to valid positions
H, W = depth.shape
v_coords = torch.arange(0, H, 4, dtype=torch.long)
u_coords = torch.arange(0, W, 4, dtype=torch.long)
v_grid, u_grid = torch.meshgrid(v_coords, u_coords, indexing='ij')
depth_valid = depth[v_grid, u_grid].reshape(-1) > 0.1

scales_basic = scales_basic[depth_valid]
scales_adaptive = scales_adaptive[depth_valid]

print(f"Basic scales: mean={scales_basic[:, 0].mean():.4f}, std={scales_basic[:, 0].std():.4f}")
print(f"Adaptive scales: mean={scales_adaptive[:, 0].mean():.4f}, std={scales_adaptive[:, 0].std():.4f}")

In [ ]:
# Visualize scale distribution

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram comparison
axes[0].hist(scales_basic[:, 0].numpy(), bins=50, alpha=0.7, label='Basic')
axes[0].hist(scales_adaptive[:, 0].numpy(), bins=50, alpha=0.7, label='Adaptive')
axes[0].set_xlabel('Scale')
axes[0].set_ylabel('Count')
axes[0].set_title('Scale Distribution')
axes[0].legend()
axes[0].set_xlim(0, 0.05)

# Scale vs Depth
depth_flat = depth[v_grid, u_grid].reshape(-1)[depth_valid]
axes[1].scatter(depth_flat.numpy(), scales_basic[:, 0].numpy(), s=1, alpha=0.3, label='Basic')
axes[1].scatter(depth_flat.numpy(), scales_adaptive[:, 0].numpy(), s=1, alpha=0.3, label='Adaptive')
axes[1].set_xlabel('Depth (m)')
axes[1].set_ylabel('Scale')
axes[1].set_title('Scale vs Depth')
axes[1].legend()

# Scale map visualization
scale_map = torch.zeros(depth.shape)
scale_flat = scales_adaptive[:, 0]
h_sub, w_sub = len(v_coords), len(u_coords)
idx = 0
for i, v in enumerate(v_coords):
    for j, u in enumerate(u_coords):
        if depth[v, u] > 0.1:
            scale_map[v, u] = scale_flat[idx]
            idx += 1

im = axes[2].imshow(scale_map.numpy(), cmap='hot')
axes[2].set_title('Adaptive Scale Map')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], label='Scale')

plt.tight_layout()
plt.show()

## 5. Color Initialization (RGB to SH)

Gaussian colors are stored as **Spherical Harmonics (SH)** coefficients. For initialization:

- Use only the DC (degree 0) component
- Higher SH degrees are initialized to zero
- SH degree 0 coefficient: $c_0 = \frac{\text{RGB} - 0.5}{C_0}$ where $C_0 = 0.28209...$

In [ ]:
# SH Constants
SH_C0 = 0.28209479177387814
SH_C1 = 0.4886025119029199
SH_C2_0 = 1.0925484305920792
SH_C2_1 = 0.31539156525252005
SH_C2_2 = 0.5462742152960396


def rgb_to_sh(rgb: torch.Tensor) -> torch.Tensor:
    """
    Convert RGB color to SH DC coefficient.
    
    Args:
        rgb: [N, 3] RGB colors in [0, 1]
    
    Returns:
        sh_dc: [N, 1, 3] SH DC coefficients
    """
    # RGB is evaluated at: color = SH_C0 * sh_dc + 0.5
    # So: sh_dc = (RGB - 0.5) / SH_C0
    sh_dc = (rgb - 0.5) / SH_C0
    return sh_dc.unsqueeze(1)  # [N, 1, 3]


def sh_to_rgb(sh_dc: torch.Tensor) -> torch.Tensor:
    """
    Convert SH DC coefficient back to RGB.
    
    Args:
        sh_dc: [N, 1, 3] SH DC coefficients
    
    Returns:
        rgb: [N, 3] RGB colors
    """
    rgb = SH_C0 * sh_dc.squeeze(1) + 0.5
    return torch.clamp(rgb, 0, 1)


def initialize_sh_features(
    colors: torch.Tensor,  # [N, 3] RGB
    sh_degree: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Initialize SH features from RGB colors.
    
    Args:
        colors: [N, 3] RGB colors in [0, 1]
        sh_degree: Maximum SH degree
    
    Returns:
        features_dc: [N, 1, 3] DC component
        features_rest: [N, K, 3] Higher-order components (zeros)
    """
    N = colors.shape[0]
    
    # DC component
    features_dc = rgb_to_sh(colors)
    
    # Higher-order components (initialized to zero)
    n_sh_coeffs = (sh_degree + 1) ** 2 - 1  # Exclude DC
    features_rest = torch.zeros(N, max(n_sh_coeffs, 1), 3)
    
    return features_dc, features_rest


# Test color conversion
test_rgb = torch.tensor([
    [1.0, 0.0, 0.0],  # Red
    [0.0, 1.0, 0.0],  # Green
    [0.0, 0.0, 1.0],  # Blue
    [0.5, 0.5, 0.5],  # Gray
    [1.0, 1.0, 1.0],  # White
    [0.0, 0.0, 0.0],  # Black
])

sh_dc = rgb_to_sh(test_rgb)
rgb_reconstructed = sh_to_rgb(sh_dc)

print("Color Conversion Test:")
print("-" * 50)
for i in range(len(test_rgb)):
    orig = test_rgb[i].numpy()
    recon = rgb_reconstructed[i].numpy()
    error = np.abs(orig - recon).max()
    print(f"  RGB {orig} -> SH {sh_dc[i, 0].numpy()} -> RGB {recon} | Error: {error:.6f}")

In [ ]:
# Visualize SH color space

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Create color gradient
n_colors = 100
rgb_gradient = torch.zeros(n_colors, 3)
rgb_gradient[:, 0] = torch.linspace(0, 1, n_colors)  # Red ramp

sh_gradient = rgb_to_sh(rgb_gradient)

# RGB vs SH coefficient
axes[0].plot(rgb_gradient[:, 0].numpy(), sh_gradient[:, 0, 0].numpy(), 'r-', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('RGB Red Value')
axes[0].set_ylabel('SH DC Coefficient')
axes[0].set_title('RGB to SH Mapping')
axes[0].grid(True, alpha=0.3)

# Color wheel in RGB
angles = np.linspace(0, 2*np.pi, 360)
rgb_wheel = np.zeros((360, 3))
for i, a in enumerate(angles):
    # HSV to RGB (simplified)
    h = a / (2 * np.pi) * 6
    sector = int(h)
    frac = h - sector
    
    if sector == 0:
        rgb_wheel[i] = [1, frac, 0]
    elif sector == 1:
        rgb_wheel[i] = [1-frac, 1, 0]
    elif sector == 2:
        rgb_wheel[i] = [0, 1, frac]
    elif sector == 3:
        rgb_wheel[i] = [0, 1-frac, 1]
    elif sector == 4:
        rgb_wheel[i] = [frac, 0, 1]
    else:
        rgb_wheel[i] = [1, 0, 1-frac]

sh_wheel = rgb_to_sh(torch.from_numpy(rgb_wheel).float()).numpy()

axes[1].scatter(sh_wheel[:, 0, 0], sh_wheel[:, 0, 1], c=rgb_wheel, s=20)
axes[1].set_xlabel('SH Red')
axes[1].set_ylabel('SH Green')
axes[1].set_title('Color Wheel in SH Space (R vs G)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

# SH coefficient ranges
n_samples = 1000
random_rgb = torch.rand(n_samples, 3)
random_sh = rgb_to_sh(random_rgb)

axes[2].hist(random_sh[:, 0, 0].numpy(), bins=50, alpha=0.7, label='R', color='red')
axes[2].hist(random_sh[:, 0, 1].numpy(), bins=50, alpha=0.7, label='G', color='green')
axes[2].hist(random_sh[:, 0, 2].numpy(), bins=50, alpha=0.7, label='B', color='blue')
axes[2].set_xlabel('SH Coefficient Value')
axes[2].set_ylabel('Count')
axes[2].set_title('SH DC Coefficient Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nSH DC range for RGB [0,1]: [{-0.5/SH_C0:.3f}, {0.5/SH_C0:.3f}]")

## 6. Opacity Initialization

Initial opacity affects convergence speed:

- **Too low**: Gaussians invisible, hard to optimize
- **Too high**: Overlapping Gaussians block each other
- **Recommended**: 0.5 (middle ground)

Opacity is stored as **logit** for unconstrained optimization:
$$
\text{logit}(\alpha) = \log\left(\frac{\alpha}{1-\alpha}\right)
$$

In [ ]:
def opacity_to_logit(opacity: float) -> float:
    """Convert opacity [0,1] to logit (-inf, inf)."""
    return np.log(opacity / (1 - opacity))


def logit_to_opacity(logit: float) -> float:
    """Convert logit to opacity."""
    return 1 / (1 + np.exp(-logit))


def initialize_opacity(
    n_gaussians: int,
    initial_opacity: float = 0.5,
) -> torch.Tensor:
    """
    Initialize opacity values as logits.
    
    Args:
        n_gaussians: Number of Gaussians
        initial_opacity: Target opacity value in [0, 1]
    
    Returns:
        opacity_logit: [N, 1] logit values
    """
    logit = opacity_to_logit(initial_opacity)
    return torch.full((n_gaussians, 1), logit)


# Test opacity conversion
test_opacities = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]

print("Opacity Conversion:")
print("-" * 40)
for alpha in test_opacities:
    logit = opacity_to_logit(alpha)
    recovered = logit_to_opacity(logit)
    print(f"  α={alpha:.2f} -> logit={logit:+.3f} -> α={recovered:.6f}")

In [ ]:
# Visualize opacity sigmoid

logits = np.linspace(-6, 6, 200)
opacities = 1 / (1 + np.exp(-logits))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sigmoid curve
axes[0].plot(logits, opacities, 'b-', linewidth=2)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Mark common initial values
for alpha in [0.1, 0.5, 0.9]:
    logit = opacity_to_logit(alpha)
    axes[0].plot(logit, alpha, 'ro', markersize=10)
    axes[0].annotate(f'α={alpha}', (logit, alpha), xytext=(logit+0.5, alpha+0.05))

axes[0].set_xlabel('Logit')
axes[0].set_ylabel('Opacity')
axes[0].set_title('Sigmoid: logit → opacity')
axes[0].grid(True, alpha=0.3)

# Gradient of sigmoid (useful for understanding optimization)
grad = opacities * (1 - opacities)
axes[1].plot(logits, grad, 'g-', linewidth=2)
axes[1].set_xlabel('Logit')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Sigmoid Gradient (∂α/∂logit)')
axes[1].grid(True, alpha=0.3)

# Mark where gradient is highest
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Max gradient at logit=0')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nInitial opacity recommendations:")
print("  - α=0.5 (logit=0): Maximum gradient, fastest convergence")
print("  - α=0.1 (logit=-2.2): Low visibility, use for dense initialization")
print("  - α=0.9 (logit=2.2): High visibility, risk of over-occlusion")

## 7. Complete Initialization Pipeline

Let's put it all together: a complete Gaussian initialization from RGB-D.

In [ ]:
@dataclass
class GaussianInitConfig:
    """Configuration for Gaussian initialization."""
    subsample: int = 4
    min_depth: float = 0.1
    max_depth: float = 10.0
    initial_opacity: float = 0.5
    scale_factor: float = 1.0
    min_scale: float = 0.001
    max_scale: float = 0.1
    sh_degree: int = 0
    use_adaptive_scale: bool = True


class GaussianInitializer:
    """
    Complete Gaussian initialization from RGB-D.
    """
    
    def __init__(self, config: GaussianInitConfig = None):
        self.config = config or GaussianInitConfig()
    
    def initialize(
        self,
        rgb: torch.Tensor,  # [3, H, W]
        depth: torch.Tensor,  # [H, W]
        intrinsics: CameraIntrinsics,
        pose: torch.Tensor = None,  # [4, 4]
    ) -> Dict[str, torch.Tensor]:
        """
        Initialize Gaussians from RGB-D frame.
        
        Returns:
            Dict with all Gaussian parameters:
            - xyz: [N, 3] positions
            - features_dc: [N, 1, 3] SH DC
            - features_rest: [N, K, 3] SH rest
            - scaling: [N, 3] log-scales
            - rotation: [N, 4] quaternions
            - opacity: [N, 1] logit-opacity
        """
        cfg = self.config
        
        # 1. Get positions and colors
        positions, colors, _ = initialize_gaussians_from_rgbd(
            rgb, depth, intrinsics, pose,
            subsample=cfg.subsample,
            min_depth=cfg.min_depth,
            max_depth=cfg.max_depth,
        )
        
        N = len(positions)
        print(f"Initialized {N} Gaussians from RGB-D frame")
        
        # 2. Estimate scales
        if cfg.use_adaptive_scale:
            scales = estimate_scales_from_depth_gradient(
                depth, intrinsics, cfg.subsample,
                cfg.min_scale, cfg.max_scale
            )
        else:
            scales = estimate_gaussian_scales(
                depth, intrinsics, cfg.subsample, cfg.scale_factor
            )
        
        # Filter to valid depth positions
        H, W = depth.shape
        v_coords = torch.arange(0, H, cfg.subsample, dtype=torch.long)
        u_coords = torch.arange(0, W, cfg.subsample, dtype=torch.long)
        v_grid, u_grid = torch.meshgrid(v_coords, u_coords, indexing='ij')
        depth_flat = depth[v_grid, u_grid].reshape(-1)
        valid = (depth_flat > cfg.min_depth) & (depth_flat < cfg.max_depth)
        scales = scales[valid]
        
        # 3. Initialize colors (RGB to SH)
        features_dc, features_rest = initialize_sh_features(colors, cfg.sh_degree)
        
        # 4. Initialize rotation (identity quaternions)
        rotation = torch.zeros(N, 4)
        rotation[:, 0] = 1.0  # w = 1
        
        # 5. Initialize opacity
        opacity = initialize_opacity(N, cfg.initial_opacity)
        
        # 6. Convert scales to log-space
        log_scales = torch.log(scales)
        
        return {
            'xyz': positions,
            'features_dc': features_dc,
            'features_rest': features_rest,
            'scaling': log_scales,
            'rotation': rotation,
            'opacity': opacity,
        }
    
    def summary(self, params: Dict[str, torch.Tensor]):
        """Print initialization summary."""
        print("\nGaussian Initialization Summary:")
        print("=" * 50)
        print(f"  Number of Gaussians: {len(params['xyz']):,}")
        print(f"\n  Positions:")
        print(f"    X: [{params['xyz'][:, 0].min():.3f}, {params['xyz'][:, 0].max():.3f}]")
        print(f"    Y: [{params['xyz'][:, 1].min():.3f}, {params['xyz'][:, 1].max():.3f}]")
        print(f"    Z: [{params['xyz'][:, 2].min():.3f}, {params['xyz'][:, 2].max():.3f}]")
        
        scales = torch.exp(params['scaling'])
        print(f"\n  Scales:")
        print(f"    Mean: {scales.mean():.6f}")
        print(f"    Range: [{scales.min():.6f}, {scales.max():.6f}]")
        
        opacity = torch.sigmoid(params['opacity'])
        print(f"\n  Opacity:")
        print(f"    Mean: {opacity.mean():.3f}")
        print(f"    Range: [{opacity.min():.3f}, {opacity.max():.3f}]")
        
        print(f"\n  Memory estimate:")
        total_params = sum(p.numel() for p in params.values())
        memory_mb = total_params * 4 / (1024 * 1024)  # float32
        print(f"    Total parameters: {total_params:,}")
        print(f"    Memory (float32): {memory_mb:.2f} MB")


# Initialize Gaussians
initializer = GaussianInitializer(GaussianInitConfig(
    subsample=4,
    initial_opacity=0.5,
    use_adaptive_scale=True,
))

gaussian_params = initializer.initialize(rgb, depth, camera)
initializer.summary(gaussian_params)

## 8. RGB-D vs Monocular Initialization

Without depth sensors, we need alternative approaches:

| Approach | Input | Pros | Cons |
|----------|-------|------|------|
| **RGB-D** | Depth sensor | Accurate, dense | Requires hardware |
| **Stereo** | Two cameras | Good accuracy | Calibration needed |
| **SfM** | Multiple RGB | No extra hardware | Slow, needs motion |
| **Learned depth** | Single RGB | Fast, no motion | Scale ambiguity |

### MonoGS Approach

For monocular SLAM, MonoGS uses:
1. **Depth prediction** network (e.g., MiDaS, DPT)
2. **Scale alignment** using prior knowledge
3. **Geometric regularization** to maintain consistency

In [ ]:
def simulate_monocular_depth(
    rgb: torch.Tensor,
    gt_depth: torch.Tensor,
    scale_factor: float = 1.0,
    noise_std: float = 0.1,
) -> torch.Tensor:
    """
    Simulate monocular depth estimation.
    
    Adds noise and scale ambiguity to ground truth depth.
    """
    # Add noise
    noise = torch.randn_like(gt_depth) * noise_std * gt_depth
    noisy_depth = gt_depth + noise
    
    # Apply unknown scale factor
    scaled_depth = noisy_depth * scale_factor
    
    # Clamp to valid range
    return torch.clamp(scaled_depth, 0.1, 10.0)


def align_monocular_depth(
    predicted_depth: torch.Tensor,
    reference_depth: torch.Tensor,
    mask: torch.Tensor = None,
) -> Tuple[torch.Tensor, float, float]:
    """
    Align predicted depth to reference using least squares.
    
    Finds scale and shift: aligned = scale * predicted + shift
    
    Returns:
        aligned_depth, scale, shift
    """
    if mask is None:
        mask = (predicted_depth > 0) & (reference_depth > 0)
    
    pred_valid = predicted_depth[mask].reshape(-1)
    ref_valid = reference_depth[mask].reshape(-1)
    
    # Least squares: ref = scale * pred + shift
    # [pred, 1] @ [scale, shift]^T = ref
    A = torch.stack([pred_valid, torch.ones_like(pred_valid)], dim=1)
    b = ref_valid
    
    # Solve using pseudoinverse
    solution = torch.linalg.lstsq(A, b).solution
    scale = solution[0].item()
    shift = solution[1].item()
    
    aligned = scale * predicted_depth + shift
    
    return aligned, scale, shift


# Simulate monocular depth
mono_depth = simulate_monocular_depth(rgb, depth, scale_factor=0.7, noise_std=0.15)

# Align to ground truth
aligned_depth, scale, shift = align_monocular_depth(mono_depth, depth)

# Compute errors
mono_error = (mono_depth - depth).abs().mean()
aligned_error = (aligned_depth - depth).abs().mean()

print(f"Monocular Depth Alignment:")
print(f"  Scale: {scale:.3f}")
print(f"  Shift: {shift:.3f}")
print(f"  MAE before alignment: {mono_error:.4f} m")
print(f"  MAE after alignment: {aligned_error:.4f} m")

In [ ]:
# Visualize monocular depth vs RGB-D

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Ground truth depth
im0 = axes[0, 0].imshow(depth.numpy(), cmap='viridis')
axes[0, 0].set_title('Ground Truth Depth')
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0])

# Simulated monocular depth
im1 = axes[0, 1].imshow(mono_depth.numpy(), cmap='viridis')
axes[0, 1].set_title('Monocular Depth (Unaligned)')
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1])

# Aligned monocular depth
im2 = axes[0, 2].imshow(aligned_depth.numpy(), cmap='viridis')
axes[0, 2].set_title('Monocular Depth (Aligned)')
axes[0, 2].axis('off')
plt.colorbar(im2, ax=axes[0, 2])

# Error maps
error_mono = (mono_depth - depth).abs().numpy()
error_aligned = (aligned_depth - depth).abs().numpy()

im3 = axes[1, 0].imshow(error_mono, cmap='hot', vmin=0, vmax=1)
axes[1, 0].set_title(f'Unaligned Error (MAE: {mono_error:.3f}m)')
axes[1, 0].axis('off')
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(error_aligned, cmap='hot', vmin=0, vmax=1)
axes[1, 1].set_title(f'Aligned Error (MAE: {aligned_error:.3f}m)')
axes[1, 1].axis('off')
plt.colorbar(im4, ax=axes[1, 1])

# Depth scatter plot
axes[1, 2].scatter(depth.numpy().flatten()[::100], mono_depth.numpy().flatten()[::100], 
                   s=1, alpha=0.3, label='Unaligned')
axes[1, 2].scatter(depth.numpy().flatten()[::100], aligned_depth.numpy().flatten()[::100], 
                   s=1, alpha=0.3, label='Aligned')
axes[1, 2].plot([0, 4], [0, 4], 'k--', label='Perfect')
axes[1, 2].set_xlabel('Ground Truth Depth')
axes[1, 2].set_ylabel('Estimated Depth')
axes[1, 2].set_title('Depth Correlation')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary

### Key Takeaways

1. **Position initialization**: Unproject depth to 3D using camera intrinsics
2. **Scale estimation**: Use depth-based or gradient-adaptive scales
3. **Color initialization**: Convert RGB to SH DC coefficients
4. **Opacity initialization**: Start with 0.5 for best gradient flow
5. **Monocular challenges**: Scale ambiguity requires alignment

### Best Practices

| Parameter | Recommendation | Reason |
|-----------|----------------|--------|
| Subsample | 4-8 | Balance coverage vs memory |
| Initial opacity | 0.5 | Maximum gradient |
| Scale factor | 0.5-1.0 | Cover pixel footprint |
| SH degree | 0 | Speed, add later |

---

## What's Next?

**[04_camera_tracking.ipynb](./04_camera_tracking.ipynb)** - Camera tracking using Gaussian rendering

---

## References

1. SplaTAM: https://spla-tam.github.io/
2. MonoGS: https://rmurai.co.uk/projects/GaussianSplattingSLAM/
3. MiDaS depth estimation: https://github.com/isl-org/MiDaS
4. Pinhole camera model: https://en.wikipedia.org/wiki/Pinhole_camera_model